# Causal Circuit Discovery & Steering
## OLMo-3 7B + OLMo-3.1 32B

Runs H18-H21 on both models, with generation examples for every steering method.
Estimated runtime: ~70 min total on A100.

In [ ]:
import subprocess, sys, os

if not os.path.exists('/content/clin-syco'):
    subprocess.run(['git', 'clone', 'https://github.com/elliott-leow/clin-syco.git', '/content/clin-syco'], check=True)

os.chdir('/content/clin-syco')
subprocess.run(['git', 'pull', '--ff-only'], check=False)

import importlib
for key in list(sys.modules.keys()):
    if key.startswith('experiments.') or key.startswith('pals.'):
        del sys.modules[key]

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], check=True)

import torch
print(f'PyTorch {torch.__version__}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
import gc, json, os, time, torch
import torch.nn.functional as F
from datetime import datetime, timezone

sys.path.insert(0, '/content/clin-syco/src')
sys.path.insert(0, '/content/clin-syco')

STIMULI_DIR = '/content/clin-syco/stimuli'
DEVICE = 'cuda'

with open(os.path.join(STIMULI_DIR, 'cognitive_distortions_augmented.json')) as f:
    all_clinical = json.load(f)
train_clinical = all_clinical[:50]
test_clinical = all_clinical[50:]
clinical = all_clinical  # backward compat
print(f'Stimuli: {len(all_clinical)} total, {len(train_clinical)} train, {len(test_clinical)} test')

def cleanup():
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

def vram():
    return torch.cuda.memory_allocated()/1e9 if torch.cuda.is_available() else 0

def clear_hf_cache(model_id):
    import shutil
    cache_dir = os.path.join(os.path.expanduser('~'), '.cache/huggingface/hub',
                             'models--' + model_id.replace('/', '--'))
    if os.path.exists(cache_dir):
        sz = sum(os.path.getsize(os.path.join(dp,f)) for dp,dn,fn in os.walk(cache_dir) for f in fn)
        shutil.rmtree(cache_dir)
        print(f'  Cleared {sz/1e9:.1f} GB cache')

def generate_examples(model, tokenizer, direction, steer_layers, prompts, alpha=6.0):
    """Generate baseline, single-layer steered, and multi-layer steered text."""
    device = next(model.parameters()).device
    dtype = next(model.parameters()).dtype
    mid = steer_layers[len(steer_layers)//2]

    results = []
    for s in prompts:
        ids = tokenizer.encode(s['user_prompt'], return_tensors='pt').to(device)

        # baseline
        with torch.no_grad():
            out = model.generate(ids, max_new_tokens=80, do_sample=False)
        baseline = tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True)

        # single-layer
        vec = direction[mid].to(device=device, dtype=dtype)
        def hook_s(mod, inp, out, v=vec, a=alpha):
            h = out[0] if isinstance(out, tuple) else out
            h = h.clone(); h[:, -1, :] -= a * v
            return (h,) + out[1:] if isinstance(out, tuple) else h
        handle = model.model.layers[mid].register_forward_hook(hook_s)
        with torch.no_grad():
            out_s = model.generate(ids, max_new_tokens=80, do_sample=False)
        handle.remove()
        steered = tokenizer.decode(out_s[0][ids.shape[1]:], skip_special_tokens=True)

        # multi-layer
        handles = []
        for sl in steer_layers:
            sv = direction[sl].to(device=device, dtype=dtype)
            def make_h(v, a=alpha*0.6):
                def fn(mod, inp, out):
                    h = out[0] if isinstance(out, tuple) else out
                    h = h.clone(); h[:, -1, :] -= a * v
                    return (h,) + out[1:] if isinstance(out, tuple) else h
                return fn
            handles.append(model.model.layers[sl].register_forward_hook(make_h(sv)))
        with torch.no_grad():
            out_m = model.generate(ids, max_new_tokens=80, do_sample=False)
        for h in handles: h.remove()
        multi = tokenizer.decode(out_m[0][ids.shape[1]:], skip_special_tokens=True)

        results.append({'subcategory': s.get('subcategory',''), 'prompt': s['user_prompt'],
                        'baseline': baseline, 'steered': steered, 'multi_layer': multi})
    return results

def print_examples(examples, model_name):
    print(f'\n{"="*70}')
    print(f'GENERATION EXAMPLES — {model_name}')
    print(f'{"="*70}')
    for i, ex in enumerate(examples):
        print(f'\n--- Example {i+1} [{ex["subcategory"]}] ---')
        print(f'Prompt: {ex["prompt"][:100]}...\n')
        print(f'BASELINE:    {ex["baseline"][:250]}')
        print(f'\nSTEERED:     {ex["steered"][:250]}')
        print(f'\nMULTI-LAYER: {ex["multi_layer"][:250]}')

# pick 5 diverse prompts for examples
example_prompts = [clinical[0], clinical[5], clinical[10], clinical[15], clinical[20]]
print(f'Loaded {len(clinical)} stimuli, {len(example_prompts)} for generation examples')

---
# Part 1: OLMo-3 7B DPO

Smaller model to establish the causal circuit pattern, then compare with 32B.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_7B = 'allenai/Olmo-3-7B-Instruct-DPO'
OUT_7B = '/content/clin-syco/results/olmo3_7b_causal'
os.makedirs(OUT_7B, exist_ok=True)

print(f'Loading {MODEL_7B} (fp16)...')
t0 = time.time()
model = AutoModelForCausalLM.from_pretrained(MODEL_7B, torch_dtype=torch.float16,
                                              device_map='auto', attn_implementation='sdpa')
model.eval()
tokenizer = AutoTokenizer.from_pretrained(MODEL_7B)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
n_layers = model.config.num_hidden_layers
LAYERS_7B = list(range(0, n_layers, 2)) + [n_layers-1]  # every other + last
print(f'Loaded in {time.time()-t0:.0f}s — {n_layers} layers, VRAM: {vram():.1f} GB')

In [ ]:
# H18: Causal Tracing (7B)
from experiments.h18_causal_tracing import run as run_h18

t0 = time.time()
run_h18(model, tokenizer, STIMULI_DIR, os.path.join(OUT_7B, 'h18'),
        layers=LAYERS_7B, n_stimuli=10, n_completion_tokens=3)
cleanup()
print(f'H18 done in {time.time()-t0:.0f}s')

with open(os.path.join(OUT_7B, 'h18/h18_results.json')) as f:
    h18_7b = json.load(f)
critical_7b = h18_7b['critical_layers']
print(f'Critical layers: {critical_7b}')
print(f'Clinical peak: layer {h18_7b["clinical_peak_layer"]} (recovery={h18_7b["clinical_peak_recovery"]:.3f})')

In [ ]:
# H19: Path Patching (7B)
from experiments.h19_path_patching import run as run_h19

t0 = time.time()
run_h19(model, tokenizer, STIMULI_DIR, os.path.join(OUT_7B, 'h19'),
        layers=LAYERS_7B, n_stimuli=5, critical_layers=critical_7b[:6])
cleanup()
print(f'H19 done in {time.time()-t0:.0f}s')

In [ ]:
# Steering directions for generation examples (7B)
from pals.extraction import batch_extract_contrastive
from pals.directions import compute_contrastive_direction

# Use H18 causal tracing to pick layers with highest recovery
recovery_7b = h18_7b['clinical_recovery_by_layer']
ranked_layers_7b = sorted(recovery_7b.items(), key=lambda x: -x[1])
steer_layers_7b = [int(l) for l, _ in ranked_layers_7b[:4]]
steer_layers_7b.sort()
print(f'Steering layers (top 4 by H18 causal recovery): {steer_layers_7b}')
print(f'Computing directions...')
pos, neg = batch_extract_contrastive(model, tokenizer, clinical[:15],
    'sycophantic_completion', 'therapeutic_completion', layers=steer_layers_7b, desc='Steering')
dir_7b = compute_contrastive_direction(pos, neg)
del pos, neg; cleanup()

examples_7b = generate_examples(model, tokenizer, dir_7b, steer_layers_7b, example_prompts, alpha=6.0)
print_examples(examples_7b, 'OLMo-3 7B DPO')

In [ ]:
# H20: Circuit Steering comparison (7B)
from experiments.h20_circuit_steering import run as run_h20

t0 = time.time()
run_h20(model, tokenizer, STIMULI_DIR, os.path.join(OUT_7B, 'h20'),
        layers=LAYERS_7B, n_stimuli=10, critical_layers=steer_layers_7b, alphas=[2.0, 4.0, 8.0])
cleanup()
print(f'H20 done in {time.time()-t0:.0f}s')

In [ ]:
# H21: Nonlinear Steering (7B)
from experiments.h21_nonlinear_steering import run as run_h21

t0 = time.time()
run_h21(model, tokenizer, STIMULI_DIR, os.path.join(OUT_7B, 'h21'),
        layers=LAYERS_7B, n_stimuli=15, hidden_size=32, epochs=30,
        steer_layer=steer_layers_7b[len(steer_layers_7b)//2])
cleanup()
print(f'H21 done in {time.time()-t0:.0f}s')

In [ ]:
# Free 7B
del model, tokenizer, dir_7b
cleanup()
clear_hf_cache(MODEL_7B)
print(f'7B freed. VRAM: {vram():.1f} GB')

---
# Part 2: OLMo-3.1 32B DPO

Same experiments at 32B scale. Key question: does the causal circuit scale the same way?

In [ ]:
MODEL_32B = 'allenai/Olmo-3.1-32B-Instruct-DPO'
OUT_32B = '/content/clin-syco/results/olmo3_32b_fp16_causal'
os.makedirs(OUT_32B, exist_ok=True)

print(f'Loading {MODEL_32B} (fp16)...')
t0 = time.time()
model = AutoModelForCausalLM.from_pretrained(MODEL_32B, torch_dtype=torch.float16,
                                              device_map='auto', attn_implementation='sdpa')
model.eval()
tokenizer = AutoTokenizer.from_pretrained(MODEL_32B)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
n_layers = model.config.num_hidden_layers
LAYERS_32B = list(range(0, n_layers, 4)) + [n_layers-1]
print(f'Loaded in {time.time()-t0:.0f}s — {n_layers} layers, VRAM: {vram():.1f} GB')

In [ ]:
# H18: Causal Tracing (32B)
t0 = time.time()
run_h18(model, tokenizer, STIMULI_DIR, os.path.join(OUT_32B, 'h18'),
        layers=LAYERS_32B, n_stimuli=10, n_completion_tokens=3)
cleanup()
print(f'H18 done in {time.time()-t0:.0f}s')

with open(os.path.join(OUT_32B, 'h18/h18_results.json')) as f:
    h18_32b = json.load(f)
critical_32b = h18_32b['critical_layers']
print(f'Critical layers: {critical_32b}')
print(f'Clinical peak: layer {h18_32b["clinical_peak_layer"]} (recovery={h18_32b["clinical_peak_recovery"]:.3f})')

In [ ]:
# H19: Path Patching (32B)
t0 = time.time()
run_h19(model, tokenizer, STIMULI_DIR, os.path.join(OUT_32B, 'h19'),
        layers=LAYERS_32B, n_stimuli=5, critical_layers=critical_32b[:6])
cleanup()
print(f'H19 done in {time.time()-t0:.0f}s')

In [ ]:
# Steering directions + generation examples (32B)
recovery_32b = h18_32b['clinical_recovery_by_layer']
ranked_layers_32b = sorted(recovery_32b.items(), key=lambda x: -x[1])
steer_layers_32b = [int(l) for l, _ in ranked_layers_32b[:4]]
steer_layers_32b.sort()
print(f'Steering layers (top 4 by H18 causal recovery): {steer_layers_32b}')
print(f'Computing directions...')
pos, neg = batch_extract_contrastive(model, tokenizer, clinical[:15],
    'sycophantic_completion', 'therapeutic_completion', layers=steer_layers_32b, desc='Steering')
dir_32b = compute_contrastive_direction(pos, neg)
del pos, neg; cleanup()

examples_32b = generate_examples(model, tokenizer, dir_32b, steer_layers_32b, example_prompts, alpha=6.0)
print_examples(examples_32b, 'OLMo-3.1 32B DPO')

In [ ]:
# H20: Circuit Steering (32B)
t0 = time.time()
run_h20(model, tokenizer, STIMULI_DIR, os.path.join(OUT_32B, 'h20'),
        layers=LAYERS_32B, n_stimuli=10, critical_layers=steer_layers_32b, alphas=[2.0, 4.0, 8.0])
cleanup()
print(f'H20 done in {time.time()-t0:.0f}s')

In [ ]:
# H21: Nonlinear Steering (32B)
t0 = time.time()
run_h21(model, tokenizer, STIMULI_DIR, os.path.join(OUT_32B, 'h21'),
        layers=LAYERS_32B, n_stimuli=15, hidden_size=32, epochs=30,
        steer_layer=steer_layers_32b[len(steer_layers_32b)//2])
cleanup()
print(f'H21 done in {time.time()-t0:.0f}s')

---
# Cross-Scale Comparison

In [ ]:
# Side-by-side comparison
print('='*70)
print('CROSS-SCALE CAUSAL TRACING COMPARISON')
print('='*70)

for name, h18 in [('7B', h18_7b), ('32B', h18_32b)]:
    n = max(int(k) for k in h18['clinical_recovery_by_layer'].keys()) + 1
    peak = h18['clinical_peak_layer']
    print(f'\n{name} ({n} layers):')
    print(f'  Clinical peak: layer {peak} ({peak/n*100:.0f}% depth), recovery={h18["clinical_peak_recovery"]:.3f}')
    print(f'  Factual peak:  layer {h18["factual_peak_layer"]} ({h18["factual_peak_layer"]/n*100:.0f}% depth)')
    print(f'  Critical layers: {h18["critical_layers"]}')

# Compare generation examples
print(f'\n{"="*70}')
print('STEERING COMPARISON: SAME PROMPTS, DIFFERENT SCALES')
print(f'{"="*70}')
for i in range(len(example_prompts)):
    e7, e32 = examples_7b[i], examples_32b[i]
    print(f'\n--- [{e7["subcategory"]}] {e7["prompt"][:80]}... ---')
    print(f'  7B baseline:    {e7["baseline"][:150]}...')
    print(f'  7B multi-steer: {e7["multi_layer"][:150]}...')
    print(f'  32B baseline:   {e32["baseline"][:150]}...')
    print(f'  32B multi-steer:{e32["multi_layer"][:150]}...')

In [ ]:
# Save all results and examples
all_results = {
    '7b': {'h18': h18_7b, 'examples': examples_7b},
    '32b': {'h18': h18_32b, 'examples': examples_32b},
}
with open('/content/clin-syco/results/causal_comparison.json', 'w') as f:
    json.dump(all_results, f, indent=2)

# Download
import shutil
zip_path = '/content/pals_causal_steering'
shutil.make_archive(zip_path, 'zip', '/content/clin-syco/results/')
print(f'Archive: {zip_path}.zip ({os.path.getsize(zip_path + ".zip") / 1e6:.1f} MB)')
try:
    from google.colab import files
    files.download(zip_path + '.zip')
except ImportError:
    print(f'Download: {zip_path}.zip')